# Títol: 12. Anàlisi del llenguatge amb spaCy

D'aquesta unitat disposes dels següents arxius:

- Els programes i arxius necessaris: [programes12-cat.zip](https://github.com/aoliverg/python/blob/master/programes12-cat/programes12-cat.zip)

---

# Instruccions prèvies per a la descàrrega/pujada d'arxius necessaris (`freeling-cat.txt`)

Per poder executar les seccions d'aquest notebook relacionades amb la descàrrega d'arxius externs o diccionaris de FreeLing (Secció 4), cal disposar dels fitxers corresponents al directori de treball.

A continuació teniu les utilitats de codi segons com esteu executant el notebook:

### Opció A: Descàrrega automàtica des de Python (Recomanat per a Colab o Local)

In [ ]:
import urllib.request
import os

# Descàrrega del diccionari de FreeLing necessari per a la secció 4
url = "https://raw.githubusercontent.com/aoliverg/python/master/programes12-cat/freeling-cat.txt"
fitxer_desti = "freeling-cat.txt"

if not os.path.exists(fitxer_desti):
    print("Descarregant 'freeling-cat.txt'...")
    try:
        urllib.request.urlretrieve(url, fitxer_desti)
        print("S'ha descarregat l'arxiu freeling-cat.txt correctament!")
    except Exception as e:
        print(f"No s'ha pogut descarregar el fitxer automàticament: {e}")
else:
    print("L'arxiu 'freeling-cat.txt' ja existeix al directori local.")

### Opció B: Pujar l'arxiu manualment si esteu a Google Colab

In [ ]:
# Només necessari si esteu executant això a Google Colab i voleu pujar l'arxiu des del vostre ordinador
try:
    from google.colab import files
    uploaded = files.upload()  # Seleccioneu 'freeling-cat.txt' o el ZIP de recursos des del vostre disc dur
except ImportError:
    print("Aquesta cel·la només s'executa a Google Colab.")

---

## 1. Introducció

### 1.1. Què és spaCy

spaCy és un conjunt d'eines de Processament del Llenguatge Natural (PLN o en anglès NLP, de Natural Language Processing) dissenyat específicament per a l'entorn de producció i l'extracció d'informació a gran escala. Permet analitzar textos en tots els nivells d'anàlisi, des de la superfície (les paraules) fins a la seva estructura profunda (les relacions gramaticals).

Les seves característiques principals són:

- **Rendiment i Velocitat**: Està escrit en Cython des de zero amb una gestió de memòria molt acurada, el que el fa extremadament ràpid.
- **Precisió Industrial**: Incorpora models d'aprenentatge profund (incloent transformadors com BERT) per aconseguir una gran precisió.
- **Multilingüe**: Suporta més de 75 llengües i ofereix models pre-entrenats per a moltes d'elles, incloses el català i el castellà.

**L'Arquitectura d'Anàlisi**

Quan passem un text per spaCy, aquest travessa una "canonada" (Pipeline) on cada pas (com ara el tokenitzador, l'etiquetador gramatical, o el parser sintàctic) afegeix una capa d'informació a l'objecte resultant. L'usuari té un control total sobre aquest pipeline i pot activar, desactivar o afegir els nivells d'anàlisi necessaris segons el projecte.

La web oficial de spaCy és: https://spacy.io/. Dedica uns minuts a explorar la web i llegeix amb atenció la introducció al sistema.

### 1.2. Com s'instal·la spaCy

La instal·lació és molt directa mitjançant pip. Abans d'instal·lar res, planteja't si vols crear un entorn virtual de Python (recomanat).

**Pas 1: Instal·lar el paquet**

Obre el teu terminal o consola i executa:

`pip install spacy`

Pas 2: Descarregar els models d'idioma

A diferència d'altres llibreries de NLP, spaCy no inclou els models lingüístics directament en el paquet per defecte per no ocupar massa espai. Has de descarregar el model específic per a l'idioma i mida que vulguis analitzar. En spaCy, els models es descarreguen habitualment per línia de comandes.

Per exemple, per descarregar el model mitjà en català, obre el terminal i executa:
Bash

`python -m spacy download ca_core_news_md`

(Nota: spaCy sol oferir diferents mides de models com sm (small), md (medium), lg (large) o trf (transformer). Per començar, el md és un bon equilibri entre precisió i velocitat). En alguns casos es pot necessitar una dependència addicional: click. Escriu:

`pip install click`

També és possible afegir als nostres programes una funció que verifiqui si el model està instal·lat i, si no, informar o intentar descarregar-lo. Podria ser així (programa-12.1-spa.py):

In [ ]:
import spacy
from spacy.cli import download  # Importem l'eina de descàrrega a l'inici

def carregar_nlp(nom_model='ca_core_news_md'):
    try:
        print(f"--- Inicialitzant el model ({nom_model})...")
        # exclude=['ner'] evita carregar el reconeixedor d'entitats si no el necessitem
        return spacy.load(nom_model, exclude=['ner'])
    except OSError:
        print(f"--- El model '{nom_model}' no s'ha trobat.")
        print("Descarregant-lo automàticament. Això pot trigar una mica...")
        
        # Descarreguem el model directament
        download(nom_model)
        
        # Tornem a intentar carregar-lo un cop descarregat
        print(f"--- Càrrega completada. Inicialitzant ({nom_model})...")
        return spacy.load(nom_model, exclude=['ner'])

# Ús del programa
nlp = carregar_nlp('ca_core_news_md')
doc = nlp("Això és una prova de càrrega automàtica.")

print(f"Text processat amb èxit: {doc.text}")

La primera vegada que executis el programa veuràs que descarrega el model:

```text
python3 programa-12.1-cat.py 
--- Inicialitzant el model (ca_core_news_md)...
--- El model 'ca_core_news_md' no s'ha trobat.
Descarregant-lo automàticament. Això pot trigar una mica...
Collecting ca-core-news-md==3.8.0
  Downloading [https://github.com/explosion/spacy-models/releases/download/ca_core_news_md-3.8.0/ca_core_news_md-3.8.0-py3-none-any.whl](https://github.com/explosion/spacy-models/releases/download/ca_core_news_md-3.8.0/ca_core_news_md-3.8.0-py3-none-any.whl) (49.2 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 MB 6.3 MB/s eta 0:00:00
Installing collected packages: ca-core-news-md
Successfully installed ca-core-news-md-3.8.0
✔ Download and installation successful
You can now load the package via spacy.load('ca_core_news_md')
--- Càrrega completada. Inicialitzant (ca_core_news_md)...
Text processat amb èxit: Això és una prova de càrrega automàtica.
```

Però si el tornes a executar veuràs que ja no el torna a descarregar.

```text
python3 programa-12.1-cat.py 
--- Inicialitzant el model (ca_core_news_md)...
Text processat amb èxit: Això és una prova de càrrega automàtica.
```

Fixa't que en el programa anterior s'ha afegit exclude=['ner']. Això serveix per no carregar el reconeixedor d'entitats anomenades (NER) si només necessites anàlisi morfològica o sintàctica bàsica, estalviant temps de càrrega i memòria.

Però si els programes els fas per a tu i ja saps que has descarregat el model via terminal, només necessites fer un simple spacy.load().

### 1.3 Un primer programa de prova

In [ ]:
import spacy

# Inicialitzem el pipeline en català (assegura't d'haver fet el python -m spacy download previament)
nlp = spacy.load('ca_core_news_md')

# El text que volem analitzar
text = "L'spaCy és una eina fantàstica per analitzar el català."

# Processem el text
doc = nlp(text)

# Comprovem que funciona imprimint el text processat
print(doc.text)

# Mostrem per pantalla les diferents anàlisis obtingudes
print(f"{'Text':<12} | {'Lema':<12} | {'Categoria (POS)':<15} | {'Sintaxi (Dep)'}")
print("-" * 60)

for token in doc:
    # token.text: La paraula original
    # token.lemma_: El lema (forma base de la paraula)
    # token.pos_: La categoria gramatical (nom, verb, adjectiu...)
    # token.dep_: La funció sintàctica dins la frase
    print(f"{token.text:<12} | {token.lemma_:<12} | {token.pos_:<15} | {token.dep_}")

A la sortida tindrem alguna cosa com aquesta:

```text
L'spaCy és una eina fantàstica per analitzar el català.
Text         | Lema         | Categoria (POS) | Sintaxi (Dep)
------------------------------------------------------------
L'           | el           | DET             | det
spaCy        | spaCy        | PROPN           | nsubj
és           | ser          | AUX             | cop
una          | un           | DET             | det
eina         | eina         | NOUN            | ROOT
fantàstica   | fantàstic    | ADJ             | amod
per          | per          | ADP             | mark
analitzar    | analitzar    | VERB            | advcl
el           | el           | DET             | det
català       | català       | NOUN            | obj
.            | .            | PUNCT           | punct
```

Aquest programa ha estat només un exemple. Durant aquest tutorial anirem aprofundint als diferents nivells d'anàlisi que ofereix aquesta llibreria.

**Exercici:** Modifica una mica el programa anterior, prova altres oracions o altres llengües. Fes que la sortida tingui la forma:

`L'|el|DET spaCy|spaCy|PROPN és|ser|AUX una|un|DET eina|eina|NOUN fantàstica|fantàstic|ADJ per|per|ADP analitzar|analitzar|VERB el|el|DET català|català|NOUN .|.|PUNCT`

o bé, si vols mantenir la funció sintàctica:

`L'|el|DET|det spaCy|spaCy|PROPN|nsubj és|ser|AUX|cop una|un|DET|det eina|eina|NOUN|ROOT fantàstica|fantàstic|ADJ|amod per|per|ADP|mark analitzar|analitzar|VERB|advcl el|el|DET|det català|català|NOUN|obj .|.|PUNCT|punct`

### 1.4. Nivells d'anàlisi en spaCy

spaCy pot oferir diversos nivells d'anàlisi i aquest tutorial s'organitza a partir d'aquests nivells:

- Segmentació
- Tokenització
- Anàlisi morfològica
- Anàlisi morfosintàctica
- Anàlisi sintàctica

### 1.5. Llengües i models disponibles en spaCy

En aquest enllaç podreu explorar totes les llengües i models disponibles a spaCy: [https://spacy.io/models](https://spacy.io/models)

## 2. Segmentació (Sentence Segmentation)

Quan treballem amb textos llargs (com ara notícies, llibres o pàgines web senceres), el primer pas lògic per poder analitzar-los correctament és dividir-los en les frases individuals que els componen. Aquest procés es coneix com a segmentació o Sentence Boundary Detection. Cal tenir en compte que no sempre un segment és una oració gramatical (per exemple un títol o un segment que contingui només expressions numèriques) i molt sovint en PLN es parla de segments en lloc d'oracions.

### 2.1. Com funciona la segmentació a spaCy?

A diferència d'aproximacions antigues que només buscaven punts (.) o signes d'interrogació (?) per tallar el text, spaCy utilitza el seu model d'aprenentatge profund per entendre l'estructura gramatical. Això li permet ser molt més precís i no confondre's amb abreviatures (com "Sr.", "Dr." o "p. ex.") ni amb punts suspensius.

Quan passes un text pel pipeline de spaCy (`doc = nlp(text)`), el sistema ja fa aquesta divisió automàticament gràcies al seu analitzador sintàctic (o parser).

Pots accedir a les frases detectades a través de la propietat `doc.sents`.

### 2.2. Programa d'exemple: Extraient frases

A continuació, farem un programa que agafa un text amb diverses frases, incloent-hi trampes com abreviatures i números amb punts, per veure com se'n surt l'spaCy (programa-12.2-cat.py):

In [ ]:
import spacy

# 1. Carreguem el model en català
nlp = spacy.load('ca_core_news_md')

# 2. Definim un text amb diverses frases i possibles trampes
text = """L'Institut d'Estudis Catalans (IEC) va ser fundat el 1907. El Dr. Pompeu Fabra va ser una figura clau en la normativització de la llengua! Avui en dia, gairebé 10.5 milions de persones el parlen (aproximadament). T'agradaria aprendre'n més?"""

# 3. Processem el text
doc = nlp(text)

# 4. Iterem sobre les frases i les imprimim
print("--- FRASES DETECTADES ---")
for i, frase in enumerate(doc.sents, 1):
    print(f"Frase {i}: {frase.text}")

Si l'executes, veuràs que spaCy sap perfectament que "Dr." no és el final d'una frase i que "10.5" és un número, tallant el text exactament on toca. T'ha de sortir alguna cosa semblant a aquesta:

```text
--- FRASES DETECTADES ---
Frase 1: L'Institut d'Estudis Catalans (IEC) va ser fundat el 1907.
Frase 2: El Dr. Pompeu Fabra va ser una figura clau en la normativització de la llengua!
Frase 3: Avui en dia, gairebé 10.5 milions de persones el parlen (aproximadament).
Frase 4: T'agradaria aprendre'n més?
```

**Exercici**: Posa a prova el segmentador canviat el text d'entrada per un més complex. Modifica el programa per a que llegeixi un arxiu de text i guardi el text segmentat en un altre arxiu de text.

### 2.3. Un detall tècnic important: Comptar frases

Si intentes fer `len(doc.sents)` per saber quantes frases hi ha en total, Python et donarà un error. Això passa perquè doc.sents no és una llista normal de Python, sinó un generador (un objecte que va calculant les frases una a una per estalviar memòria).

Si necessites comptar el nombre total de frases o accedir a una frase concreta (per exemple, la tercera frase), primer has de convertir aquest generador en una llista (programa-12.3-cat.py):

In [ ]:
import spacy

# 1. Carreguem el model en català
nlp = spacy.load('ca_core_news_md')

# 2. Definim un text amb diverses frases i possibles trampes
text = """L'Institut d'Estudis Catalans (IEC) va ser fundat el 1907. El Dr. Pompeu Fabra va ser una figura clau en la normativització de la llengua! Avui en dia, gairebé 10.5 milions de persones el parlen (aproximadament). T'agradaria aprendre'n més?"""

# 3. Processem el text
doc = nlp(text)

llista_frases = list(doc.sents)
# Ara sí que podem comptar-les
nombre_frases = len(llista_frases)
print(f"El text té un total de {nombre_frases} frases.")

# I accedir a una posició concreta (recorda que comencen per 0)
print(f"La segona frase és: {llista_frases[1].text}")

Ara a la sortida tindrem:

```text
El text té un total de 4 frases.
La segona frase és: El Dr. Pompeu Fabra va ser una figura clau en la normativització de la llengua!
```

### 2.4. Personalització de la segmentació

Tot i que el model per defecte de spaCy és molt robust perquè entén la sintaxi de la frase, a vegades treballem amb textos d'un domini específic (medicina, dret, documents tècnics) que tenen formats estranys o abreviatures molt concretes.

En aquests casos, spaCy ens permet personalitzar la segmentació de diferents maneres:

**A) Afegir noves abreviatures**

Si el problema és que tenim una abreviatura amb un punt (per exemple, "Excm." per "Excel·lentíssim") i spaCy es confon pensant que és el final de la frase, la solució més fàcil és afegir aquesta paraula a les excepcions del seu tokenitzador. Si el tokenitzador entén que "Excm." és una sola peça, el segmentador de frases ja no tallarà per aquest punt.

Veiem com fer-ho al programa programa-12.5-cat.py

In [ ]:
import spacy
from spacy.symbols import ORTH

nlp = spacy.load('ca_core_news_md')


text = "El Excm. Sr. Planell està de viatge. Tornarà aviat."
doc = nlp(text)

print("--- SEGMENTACIÓ PER DEFECTE ---")
for frase in doc.sents:
    print(frase.text)

excepcio = [{ORTH: "Excm."}]
nlp.tokenizer.add_special_case("Excm.", excepcio)

doc = nlp(text)

print("--- SEGMENTACIÓ AFEGINT ABREVIATURA ---")
for frase in doc.sents:
    print(frase.text)

Que ens donarà com a resultat:

```text
--- SEGMENTACIÓ PER DEFECTE ---
El Excm.
Sr. Planell està de viatge.
Tornarà aviat.
--- SEGMENTACIÓ AFEGINT ABREVIATURA ---
El Excm. Sr. Planell està de viatge.
Tornarà aviat.
```

**B) Crear regles personalitzades de segmentació**

A vegades volem forçar el tall d'una frase basant-nos en caràcters concrets, com un salt de línia (\n) o un punt i coma (;). Per fer-ho, podem crear un component personalitzat i afegir-lo al pipeline de spaCy just abans del parser (que és qui normalment decideix on acaben les frases).

Per forçar o prohibir el tall d'una frase abans d'una paraula concreta, utilitzem l'atribut is_sent_start (inici de frase) de cada token. (programa-12.6-cat.py)

In [ ]:
import spacy
from spacy.language import Language

nlp = spacy.load('ca_core_news_md')

# 1. Definim el nostre component de regles
@Language.component("regles_segmentacio_custom")
def segmentacio_custom(doc):
    # Iterem per tots els tokens menys l'últim
    for token in doc[:-1]: 
        # Si trobem un punt i coma o un salt de línia...
        if token.text in [";", "\n"]:
            # Marquem el següent token com l'inici d'una nova frase
            doc[token.i + 1].is_sent_start = True
    return doc

# 2. Afegim el component al pipeline, abans del parser
nlp.add_pipe("regles_segmentacio_custom", before="parser")

# 3. Provem-ho
text = "Primera frase normal. Segona frase amb punt i coma; tercera frase forçada per la regla.\nI quarta frase després del salt de línia."
doc = nlp(text)

print("--- SEGMENTACIÓ AMB REGLES CUSTOM ---")
for frase in doc.sents:
    # Utilitzem .strip() per netejar espais o salts de línia al principi/final
    print(f"- {frase.text.strip()}")

## 3. Tokenització

Un cop tenim el text dividit en frases, el pas següent és esmicolar aquesta cadena de caràcters en les unitats mínimes de significat: les paraules i els signes de puntuació. Aquestes peces individuals s'anomenen tokens, i el procés de separar-les és la tokenització.

### 3.1. Com funciona la tokenització a spaCy?

A primera vista, tokenitzar podria semblar tan senzill com tallar el text cada vegada que hi ha un espai en blanc. Però en llengües com el català, la realitat és molt més complexa:

- Apòstrofs: Què fem amb "L'spaCy"? S'ha de separar en "L'" i "spaCy", ja que funcionalment són un determinant i un nom.
- Pronoms febles i guionets: Què passa amb estructures com "apuntar-se" o "dona-li"? Cal separar el verb del pronom per poder analitzar-los bé.
- Puntuació: El punt final o la coma van enganxats a l'última paraula, però s'han de separar perquè són elements sintàctics independents.

L'avantatge de fer servir el model en català de spaCy (ca_core_news_md) és que ja incorpora totes les regles i excepcions pròpies de la nostra llengua, seguint els estàndards lingüístics de Universal Dependencies. A més, spaCy fa una tokenització "no destructiva": mai elimina els espais originals ni modifica el text font, només crea una llista de tokens que apunten a les diferents parts del text.

### 3.2. Programa d'exemple: Explorant els tokens

A continuació veurem com spaCy trosseja una frase que conté algunes d'aquestes dificultats típiques del català (apòstrofs, guionets i puntuació enganxada) (programa-12.8-cat.py):

In [ ]:
import spacy

# 1. Carreguem el model en català
nlp = spacy.load('ca_core_news_md')

# 2. Text amb apòstrofs, pronoms febles i puntuació
text = "L'estudiant va decidir apuntar-se al curs de l'IEC, oi?"

# 3. Processem el text
doc = nlp(text)

# 4. Imprimim una capçalera per veure-ho clar
print(f"{'Índex':<6} | {'Token':<12} | {'És lletra?':<12} | {'És puntuació?'}")
print("-" * 50)

# 5. Iterem sobre cada token de l'objecte 'doc'
for token in doc:
    # token.i : l'índex (posició) del token dins la frase
    # token.text : el text original del token
    # token.is_alpha : retorna True si només conté lletres
    # token.is_punct : retorna True si és un signe de puntuació
    
    print(f"{token.i:<6} | {token.text:<12} | {str(token.is_alpha):<12} | {str(token.is_punct)}")

Si executes el programa anterior, la sortida per pantalla et mostrarà exactament com el model ha pres les decisions de separació:

```text
Índex  | Token        | És lletra?   | És puntuació?
--------------------------------------------------
0      | L'           | False        | False
1      | estudiant    | True         | False
2      | va           | True         | False
3      | decidir      | True         | False
4      | apuntar      | True         | False
5      | -se          | False        | False
6      | al           | True         | False
7      | curs         | True         | False
8      | de           | True         | False
9      | l'           | False        | False
10     | IEC          | True         | False
11     | ,            | False        | True
12     | oi           | True         | False
13     | ?            | False        | True
```

Fixa't en els detalls importants:

- Ha separat correctament `L'` d'estudiant.
- Ha dividit el verb amb pronom en `apuntar` i `-se`.
- Els signes de puntuació `,` i `?` els ha aïllat a la perfecció.
- L'atribut `is_alpha` és `False` per a `L'` i `-se` precisament perquè contenen signes ortogràfics (l'apòstrof i el guionet), demostrant que spaCy sap distingir una paraula completament alfabètica d'una que conté altres caràcters.

## 4. Anàlisi Morfològica

Quan comencem a analitzar les paraules d'un text per classificar-les (saber si són noms, verbs, adjectius, etc.), és fonamental entendre la diferència entre dos conceptes clau en el Processament del Llenguatge Natural (PLN):

- L'anàlisi morfològica (sense desambiguar): Consisteix a analitzar una paraula de manera aïllada, fora de context, i llistar totes les seves possibles categories i formes. Per exemple, la paraula "sobre" pot ser un nom masculí (un sobre de paper), una preposició (sobre la taula) o un verb (espero que sobri menjar).
- L'anàlisi morfosintàctica (desambiguada o POS Tagging): Consisteix a analitzar la paraula dins del seu context (la frase) per determinar quina d'aquestes opcions és l'única correcta en aquella oració específica.

### 4.1. L'enfocament de spaCy vs. Diccionaris tradicionals

Eines històriques i molt potents com [FreeLing](https://nlp.lsi.upc.edu/freeling/node/1) funcionen en dos passos: primer consulten un diccionari gegantí per generar totes les opcions possibles (anàlisi morfològica) i, després, apliquen regles per descartar les incorrectes (desambiguació morfosintàctica).

L'spaCy funciona diferent. Com que està construït sobre xarxes neuronals profundes, no perd el temps calculant totes les opcions possibles d'una paraula aïllada. L'spaCy llegeix tota la frase de cop i prediu directament la resposta morfosintàctica correcta.

Per tant, spaCy no fa anàlisi morfològica pura. Com ho podem solucionar si en el nostre projecte necessitem veure totes les possibilitats d'una paraula? La solució és combinar la potència de tokenització de l'spaCy amb els diccionaris oberts de FreeLing.

### 4.2. Les etiquetes de FreeLing (Format EAGLES)

Els diccionaris de FreeLing són fitxers de text pla on cada línia conté tres elements: la paraula flexionada (la forma), el seu lema (l'arrel) i una etiqueta morfològica.

Aquestes etiquetes segueixen l'estàndard europeu [EAGLES](https://freeling-user-manual.readthedocs.io/en/latest/tagsets/), que és un sistema posicional. Cada lletra de l'etiqueta ens dona una informació concreta. Per exemple, l'etiqueta NCMS000:

- N: Categoria principal (Nom)
- C: Tipus (Comú)
- M: Gènere (Masculí)
- S: Nombre (Singular)

Si mirem un verb senzill com VMIP1S0 (canto):

    V: Categoria principal (Verb)
    M: Tipus (Principal / Main)
    I: Mode (Indicatiu)
    P: Temps (Present)
    1: Persona (Primera)
    S: Nombre (Singular)

(Nota: L'últim zero 0 en els verbs indica que no tenen marca de gènere, a diferència dels participis, on sí que hi hauria una M o una F).

En el següent programa (programa-12.9-cat.py), unirem el millor dels dos mons. Farem servir spaCy exclusivament per tallar l'oració correctament (separant apòstrofs, guionets i puntuació). Un cop tinguem els tokens nets, anirem a buscar cada paraula al diccionari freeling-cat.txt per extreure'n totes les interpretacions morfològiques possibles, simulant així un analitzador morfològic pur.

In [ ]:
import spacy
import os

def carregar_diccionari_morfologic(ruta_fitxer):
    """
    Llegeix el diccionari de FreeLing i en retorna un de Python.
    """
    diccionari = {}
    if not os.path.exists(ruta_fitxer):
        print(f"Error: No s'ha trobat el fitxer '{ruta_fitxer}'.")
        return diccionari

    print(f"Carregant el diccionari '{ruta_fitxer}' a la memòria...")
    with open(ruta_fitxer, 'r', encoding='utf-8') as fitxer:
        for linia in fitxer:
            parts = linia.strip().split()
            if len(parts) >= 3:
                forma, lema, etiqueta = parts[0], parts[1], parts[2]
                interpretacio = f"[{lema} - {etiqueta}]"
                
                if forma not in diccionari:
                    diccionari[forma] = []
                diccionari[forma].append(interpretacio)
                
    print(f"Diccionari carregat! Conté {len(diccionari)} formes úniques.\n")
    return diccionari


# 1. Carreguem el model de spaCy en català (només per tokenitzar)
print("--- Inicialitzant spaCy...")
try:
    nlp = spacy.load('ca_core_news_md')
except OSError:
    print("Descarregant el model ca_core_news_md...")
    from spacy.cli import download
    download('ca_core_news_md')
    nlp = spacy.load('ca_core_news_md')

# 2. Carreguem el diccionari extern
fitxer_diccionari = 'freeling-cat.txt'
analitzador_morfologic = carregar_diccionari_morfologic(fitxer_diccionari)

# 3. Només executem l'anàlisi si el diccionari s'ha carregat correctament
if analitzador_morfologic:
    
    # Definim l'oració d'entrada
    oracio = "L'estudiant va deixar el sobre groc sobre la taula, oi?"
    
    # Tokenitzem l'oració amb spaCy
    doc = nlp(oracio)
    
    print(f"Oració a analitzar: {oracio}\n")
    print(f"{'Token':<12} | {'Interpretacions Possibles'}")
    print("-" * 80)
    
    # 4. Iterem sobre els tokens que ha separat spaCy
    for token in doc:
        # Convertim a minúscules per buscar-ho al diccionari
        paraula_cerca = token.text.lower()
        
        # Cas A: La paraula és al diccionari de FreeLing
        if paraula_cerca in analitzador_morfologic:
            opcions = ", ".join(analitzador_morfologic[paraula_cerca])
            print(f"{token.text:<12} | {opcions}")
            
        # Cas B: És un signe de puntuació (ho sabem gràcies a spaCy)
        elif token.is_punct:
            print(f"{token.text:<12} | [{token.text} - PUNCT]")
            
        # Cas C: Són espais en blanc (els ignorem)
        elif token.is_space:
            continue
            
        # Cas D: La paraula no és al diccionari (neologisme, error, etc.)
        else:
            print(f"{token.text:<12} | ESCONEGUDA (No és al diccionari)")

### 4.3. El problema de l'ambigüitat

Si executes aquest programa, veuràs immediatament el gran repte del processament del llenguatge. Quan el codi arriba a la paraula "sobre", imprimeix totes les opcions del diccionari (com a nom, com a preposició i com a verb conjugat de sobrar). Només amb el diccionari, l'ordinador és incapaç de saber quina d'aquestes funcions està fent el "sobre" groc de la nostra frase.

```text
--- Inicialitzant spaCy...
Carregant el diccionari 'freeling-cat.txt' a la memòria...
Diccionari carregat! Conté 521983 formes úniques.

Oració a analitzar: L'estudiant va deixar el sobre groc sobre la taula, oi?

Token        | Interpretacions Possibles
--------------------------------------------------------------------------------
L'           | [el - DA0CS0], [el - PP3CSA0]
estudiant    | [estudiant - NCCS000], [estudiar - VMG0000]
va           | [va - AQ0MS00], [anar - VAIP3S0], [anar - VMIP3S0]
deixar       | [deixar - VMN0000]
el           | [el - DA0MS0], [el - PP3MSA0]
sobre        | [sobre - NCMS000], [sobre - SP]
groc         | [groc - AQ0MS00], [groc - NCMS000]
sobre        | [sobre - NCMS000], [sobre - SP]
la           | [la - NCMS000], [el - DA0FS0], [el - PP3FSA0], [la - I]
taula        | [taula - NCFS000], [taular - VMIP3S0], [taular - VMM02S0]
,            | [, - PUNCT]
oi           | [oi - NCMS000], [oi - I]
?            | [? - PUNCT]
```

Per resoldre aquest problema, ens cal passar al nivell següent.

## 5. Anàlisi Morfosintàctica (POS Tagging)

A l'apartat anterior hem vist el gran problema de fer servir només un diccionari: l'ambigüitat. Quan vèiem la paraula "sobre", el sistema ens donava una llista d'opcions (nom, preposició, verb...), però no sabia quina era la correcta.

A la vida real, quan parlem, una paraula només té una única funció dins d'una frase concreta. Identificar quina és aquesta funció exacta, depenent de les paraules del voltant, és el que anomenem anàlisi morfosintàctica o POS Tagging (_Part-of-Speech Tagging_).

### 5.1. Com desambigua l'spaCy?

Aquí és on brilla la intel·ligència artificial. L'spaCy no llegeix les paraules d'una en una, sinó que utilitza xarxes neuronals per llegir tota la frase sencera. En veure que darrere del primer "sobre" hi ha un adjectiu ("groc"), la xarxa dedueix matemàticament que allò ha de ser un nom. En canvi, en veure que el segon "sobre" va seguit d'un article i un nom ("la taula"), dedueix que és una preposició d'espai.

### 5.1. L'estàndard Universal Dependencies (UD)

Mentre que FreeLing utilitza l'estàndard europeu EAGLES, l'spaCy utilitza l'estàndard internacional Universal Dependencies (UD). Aquest sistema està dissenyat perquè les categories gramaticals siguin idèntiques en més de 100 idiomes diferents.

En lloc d'una sola etiqueta combinada (com NCMS000), spaCy separa la informació en dos atributs:

- token.pos_ (Categoria principal): Ens diu si és un nom (NOUN), un verb (VERB), un adjectiu (ADJ), una preposició (ADP), etc.
- token.morph (Trets morfològics): Ens detalla la flexió de la paraula (gènere, nombre, temps, persona...). Per exemple: Gender=Masc|Number=Sing.

En aquest programa (programa-12.10-cat.py) farem passar la mateixa frase trampa per l'analitzador de l'spaCy per veure com resol l'ambigüitat.

In [ ]:
import spacy

# 1. Carreguem el model en català
print("--- Inicialitzant spaCy...")
try:
    nlp = spacy.load('ca_core_news_md')
except OSError:
    from spacy.cli import download
    download('ca_core_news_md')
    nlp = spacy.load('ca_core_news_md')

# 2. La nostra oració amb la paraula polisèmica "sobre"
oracio = "L'estudiant va deixar el sobre groc sobre la taula, oi?"

# 3. Processem l'oració (aquí és on es fa la màgia de la desambiguació)
doc = nlp(oracio)

print(f"\nOració analitzada: {oracio}\n")
print(f"{'Token':<12} | {'POS (Categoria)':<15} | {'Morfologia (Trets)'}")
print("-" * 75)

# 4. Iterem sobre cada token per extreure'n l'etiqueta correcta
for token in doc:
    # token.text : La paraula original
    # token.pos_ : La categoria gramatical desambiguada (NOUN, VERB, ADP...)
    # token.morph: Els trets morfològics (Gènere, Nombre...)
    
    # Convertim l'objecte morph a text (string) per poder-lo imprimir bé. 
    # Si està buit (ex. preposicions), hi posem un guionet.
    morfologia = str(token.morph) if str(token.morph) else "-"
    
    print(f"{token.text:<12} | {token.pos_:<15} | {morfologia}")

Si executes el programa et duràs una sorpresa. La sortida per pantalla mostra un clar error en l'anàlisi:

```text
Oració analitzada: L'estudiant va deixar el sobre groc sobre la taula, oi?

Token        | POS (Categoria) | Morfologia (Trets)
---------------------------------------------------------------------------
L'           | DET             | Definite=Def|Number=Sing|PronType=Art
estudiant    | NOUN            | Number=Sing
va           | AUX             | Mood=Ind|Number=Sing|Person=3|Tense=Pres|VerbForm=Fin
deixar       | VERB            | VerbForm=Inf
el           | DET             | Definite=Def|Gender=Masc|Number=Sing|PronType=Art
sobre        | ADP             | -
groc         | ADJ             | Gender=Masc|Number=Sing
sobre        | ADP             | -
la           | DET             | Definite=Def|Gender=Fem|Number=Sing|PronType=Art
taula        | NOUN            | Gender=Fem|Number=Sing
,            | PUNCT           | PunctType=Comm
oi           | NOUN            | -
?            | PUNCT           | PunctSide=Fin|PunctType=Qest
```

El model s'ha equivocat amb el primer "sobre"! En lloc d'etiquetar-lo com a substantiu (NOUN), l'ha classificat com a preposició (ADP). En cometre aquest error, s'ha vist obligat a etiquetar la paraula amarillo com a substantiu (NOUN) per intentar buscar un sentit matemàtic a l'estructura.

Això passa perquè spaCy es basa en models estadístics i xarxes neuronals probabilístiques. En els textos d'entrenament del català, la paraula sobre apareix de forma massiva com a preposició ("parlar sobre política", "sobre la teulada"). Davant d'aquesta forta tendència estadística, el model (ca_core_news_md) ha flaquejat i no ha estat capaç de desambiguar-la correctament en aquest context concret.

En el Processament del Llenguatge Natural (PLN), els errors de desambiguació en paraules amb una categoria gramatical molt dominant (com sobre) són habituals. Per mitigar aquests errors en entorns de producció, es sol recórrer a models més grans (com ca_core_news_lg o models basats en Transformers com BERT) o bé s'introdueixen regles de correcció posteriors en el pipeline.

Exercici: Si cambies el model a ca_core_news_lg, es continua produint l'error?

**El següent nivell: Els models Transformer (_trf)**

Els models que has provat (_md i _lg) utilitzen una arquitectura clàssica de spaCy basada en xarxes neuronals convolucionals (CNN) i vectors estàtics. Són molt ràpids i lleugers, però tenen menys "llums" per entendre contextos subtils o tramposos.

Els models Transformer integran arquitectures com BERT o RoBERTa. En lloc d'assignar probabilitats rígides, calculen representacions contextuals profundes: la definició de cada paraula canvia dinàmicament segons totes les altres paraules de la frase.

El model oficial: ca_core_news_trf

spaCy ofereix models oficials basats en Transformers tant per al castellà (es_dep_news_trf, entrenat amb models com MarIA o BETO) com per al català (ca_core_news_trf, entrenat amb BERTa del projecte AINA).

Per provar-ho en el teu codi, primer has d'instal·lar l'extensió de spaCy per a Transformers i descarregar el model corresponent des de la terminal:
Bash

pip install spacy[transformers]
python -m spacy download ca_core_news_trf

I després canvies el model en l'script a ca_core_news_trf. Si ara tornes a executar el programa, hauria de desambiaguar bé:

```text
-- Inicialitzant spaCy...

Oració analitzada: L'estudiant va deixar el sobre groc sobre la taula, oi?

Token        | POS (Categoria) | Morfologia (Trets)
---------------------------------------------------------------------------
L'           | DET             | Definite=Def|Number=Sing|PronType=Art
estudiant    | NOUN            | Number=Sing
va           | AUX             | Mood=Ind|Number=Sing|Person=3|Tense=Pres|VerbForm=Fin
deixar       | VERB            | VerbForm=Inf
el           | DET             | Definite=Def|Gender=Masc|Number=Sing|PronType=Art
sobre        | NOUN            | Gender=Masc|Number=Sing
groc         | ADJ             | Gender=Masc|Number=Sing
sobre        | ADP             | -
la           | DET             | Definite=Def|Gender=Fem|Number=Sing|PronType=Art
taula        | NOUN            | Gender=Fem|Number=Sing
,            | PUNCT           | PunctType=Comm
oi           | PART            | -
?            | PUNCT           | PunctSide=Fin|PunctType=Qest
```

### 5.2. La Lematització: Trobant la forma de diccionari

Un cop l'spaCy ha desambiguat la paraula i sap exactament quina funció gramatical fa a la frase, pot fer un pas més de manera molt precisa: la lematització.

Lematitzar consisteix a extreure el "lema" o forma base d'una paraula, que és exactament la forma com la buscaríem en un diccionari:

- Pels verbs, el lema és l'infinitiu (p. ex., va -> anar, canto -> cantar).
- Pels noms i adjectius, el lema acostuma a ser el masculí singular (p. ex., nenes -> nen, groc -> groc).

L'spaCy ens proporciona aquesta informació de manera molt senzilla a través de l'atribut token.lemma_.

En aquest programa (programa-12.11-cat.py) farem passar la nostra frase trampa per l'analitzador de l'spaCy per veure com resol l'ambigüitat al vol i n'extreu l'arrel de cada paraula.

In [ ]:
import spacy

# 1. Carreguem el model en català
print("--- Inicialitzant spaCy...")
try:
    nlp = spacy.load('ca_core_news_md')
except OSError:
    from spacy.cli import download
    download('ca_core_news_md')
    nlp = spacy.load('ca_core_news_md')

# 2. La nostra oració amb la paraula polisèmica "sobre"
oracio = "L'estudiant va deixar el sobre groc sobre la taula, oi?"

# 3. Processem l'oració (es fa la desambiguació i la lematització)
doc = nlp(oracio)

print(f"\nOració analitzada: {oracio}\n")
print(f"{'Token':<12} | {'Lema':<12} | {'POS':<6} | {'Morfologia (Trets)'}")
print("-" * 80)

# 4. Iterem sobre cada token per extreure'n tota la informació
for token in doc:
    # token.text    : La paraula original
    # token.lemma_  : La forma base (lema)
    # token.pos_    : La categoria gramatical desambiguada
    # token.morph   : Els trets morfològics
    
    morfologia = str(token.morph) if str(token.morph) else "-"
    
    print(f"{token.text:<12} | {token.lemma_:<12} | {token.pos_:<6} | {morfologia}")

Si executes el programa, la sortida per pantalla serà la següent:

```text
Token        | Lema         | POS    | Morfologia (Trets)
--------------------------------------------------------------------------------
L'           | el           | DET    | Definite=Def|Number=Sing|PronType=Art
estudiant    | estudiant    | NOUN   | Number=Sing
va           | anar         | AUX    | Mood=Ind|Number=Sing|Person=3|Tense=Pres|VerbForm=Fin
deixar       | deixar       | VERB   | VerbForm=Inf
el           | el           | DET    | Definite=Def|Gender=Masc|Number=Sing|PronType=Art
sobre        | sobre        | NOUN   | Gender=Masc|Number=Sing
groc         | groc         | ADJ    | Gender=Masc|Number=Sing
sobre        | sobre        | ADP    | -
la           | el           | DET    | Definite=Def|Gender=Fem|Number=Sing|PronType=Art
taula        | taula        | NOUN   | Gender=Fem|Number=Sing
,            | ,            | PUNCT  | PunctType=Comm
oi           | oi           | INTJ   | -
?            | ?            | PUNCT  | PunctType=Qest
```

Fixa't en com influeix l'anàlisi en els resultats:

- L'impacte de l'error previ: Com que el model ca_core_news_md s'ha confós arrossegat per l'estadística i ha etiquetat el primer sobre com a preposició (ADP), el seu lema es queda simplement com a sobre. Canvia de model per veure si se soluciona aquest problema i lematitza bé.
- Lematització verbal correcta: El model detecta sense problemes que la forma conjugada va deixar prové de l'infinitiu deixar (i de l'auxiliar anar).
- Articles: Sap perfectament que tant L' com la tenen com a lema comú la forma base del determinant article masculí singular: el.

## 6. Anàlisi Sintàctica

Un cop sabem què és cada paraula (morfosintaxi) i d'on ve (lema), el següent pas és entendre com es relacionen les paraules entre elles per formar frases. Això és l'anàlisi sintàctica. En el món del PLN, hi ha dues maneres principals de representar aquesta estructura:

- Anàlisi de Dependències (spaCy): Es basa en fletxes que connecten paraules. Una paraula depèn d'una altra (ex: l'adjectiu depèn del nom). És un model molt ràpid i ideal per a l'extracció de dades.
- Anàlisi de Constituents (Benepar): Es basa en la jerarquia de sintagmes, és la clàssica anàlisi sintàctica en arbres. És un model més lent però molt útil per a estudis gramaticals profunds.

### 6.1. Anàlisi de Dependències amb spaCy

L'spaCy identifica quina paraula és el "cap" (head) i quina el "dependent". L'etiqueta de la relació (subjecte, objecte, etc.) es troba a l'atribut token.dep_. (programa-12.12-cat.py)

In [ ]:
import spacy

nlp = spacy.load('ca_core_news_md')
doc = nlp("L'estudiant llegeix un llibre molt antic.")

print(f"{'Token':<12} | {'Relació':<10} | {'Head':<12}")
print("-" * 40)
for token in doc:
    print(f"{token.text:<12} | {token.dep_:<10} | {token.head.text:<12}")

Que ofereix el següent resultat:

```text
Token        | Relació    | Head        
----------------------------------------
L'           | det        | estudiant   
estudiant    | nsubj      | llegeix     
llegeix      | ROOT       | llegeix     
un           | det        | llibre      
llibre       | obj        | llegeix     
molt         | advmod     | antic       
antic        | amod       | llibre      
.            | punct      | llegeix     
```

Nota: La paraula principal de la frase, que no depèn de cap altra, rep l'etiqueta ROOT (normalment el verb principal).

### 6.2. Anàlisi de Sintagmes (L'agrupació de paraules)

Tot i que spaCy no fa una anàlisi de "constituents" completa (aquella que posa tota la frase dins de caixes jeràrquiques), sí que ens permet extreure de manera molt eficient els Sintagmes Nominals (Noun Chunks).

Aquests són els grups de paraules que tenen un nom com a nucli (incloent els seus determinants i adjectius).

Programa d'exemple (Sintagmes Nominals) (programa-12.13-cat.py)

In [ ]:
import spacy

nlp = spacy.load('ca_core_news_md')
doc = nlp("L'estudiant de la universitat llegeix un llibre molt antic.")

print("--- Sintagmes Nominals detectats ---")
for chunk in doc.noun_chunks:
    # Mostrem el text del sintagma, el seu nucli i la seva funció
    print(f"Sintagma: {chunk.text:<30} | Nucli: {chunk.root.text}")

Que proporciona la següent sortida:

```text
--- Sintagmas Nominales detectats ---
Sintagma: L'estudiant de la universitat  | Nucli: estudiant
Sintagma: un llibre molt antic            | Nucli: llibre
```
 
Visualització Gràfica (displaCy)

spaCy inclou un motor visual que ens permet generar un diagrama de fletxes (programa-12.14-cat.py).

In [ ]:
import spacy
from spacy import displacy
from pathlib import Path

nlp = spacy.load('ca_core_news_md')
doc = nlp("L'estudiant de la universitat llegeix un llibre molt antic.")

# Rendim el gràfic directament en el Jupyter Notebook (si s'executa interactivament)
displacy.render(doc, style="dep", jupyter=True)

# Opcionalment també el guardem en un fitxer HTML per obrir-lo amb un navegador extern
svg = displacy.render(doc, style="dep", jupyter=False)
Path("arbre_sintactic.html").write_text(svg, encoding="utf-8")
print("\nTambé s'ha guardat el fitxer 'arbre_sintactic.html' al directori local.")

Recorda que el gràfic s'ha generat en l'arxiu html arbre_sintactic.html. Per visualitzar-lo el pots obrir amb qualsevol navegador. Tindrà un aspecte com aquest:

<img width="2212" height="300" alt="image" src="https://github.com/user-attachments/assets/11e63bdf-e672-4416-9a7d-a5d2dda216bd" />

## 7. Altres tipus d'anàlisi i aplicacions

Fins aquí hem vist com podem implementar els diferents nivells d'anàlisi en spaCy. A continuació us proposo algunes anàlisis alternatives i algunes aplicacions concretes de les anàlisis que hem vist fins ara. Explora almenys una d'aquestes seccions. Poden ser inspiració per al projecte de programació que hauràs de desenvolupar més endavant.

- [Complement A: Reconeixement d'Entitats Nominals (NER)](https://github.com/aoliverg/python/wiki/Complement-A:-Reconeixement-d'Entitats-Nominals-(NER))
- [Complement B: Anàlisi de sentiment (Sentiment Analysis)](https://github.com/aoliverg/python/wiki/Complement-B:-An%C3%A0lisi-de-sentiment-(Sentiment-Analysis))